# EP00｜按下送出之後，發生了什麼？

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chainchen1103/llm-from-scratch/blob/main/notebooks/ep00/ep00_next_token.ipynb)

這份 notebook 對應網站上的 [EP00 圖文教學](https://chainchen1103.github.io/llm-from-scratch/episodes/ep00/)。我們會用一個小型開源模型 Qwen2.5-0.5B，親手看看：

1. 文字怎麼被切成 token
2. 模型輸出的機率分佈長什麼樣子
3. 自己寫一個「一次產生一個 token」的生成迴圈，並調整溫度 temperature
4. 偷看殘差流 residual stream：每一層之後，模型「最想接哪個字」

## 0. 準備

Colab 已經內建 `torch` 與 `transformers`。如果在自己的電腦上執行，請先安裝這兩個套件。

In [ ]:
# %pip install -q torch transformers

import torch
import transformers

print("torch", torch.__version__)
print("transformers", transformers.__version__)

載入模型。第一次執行會下載約 1 GB 的檔案，需要等一下。

> 這裡用的是**預訓練版**，最適合觀察「猜下一個 token」。不同版本的套件、不同的數值精度，算出來的機率可能和網站上的表格有些微差異，這是正常的。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)
model.eval() 

cfg = model.config
print(f"層數：{cfg.num_hidden_layers}，每個 token 的向量長度 d：{cfg.hidden_size}，詞彙表大小：{cfg.vocab_size}")

## 1. 看 token 長什麼樣子

模型看到的不是「字」，而是 token。執行看看下面幾句話各被切成幾個 token。

In [ ]:
texts = ["今天天氣很", "今天天气很", "The weather today is very", "機器學習", "Transformer"]

for text in texts:
    ids = tok(text, return_tensors="pt").input_ids   # "pt"：回傳 PyTorch 的張量 tensor，先想成多維陣列
    pieces = [tok.decode([i]) for i in ids[0]]
    print(f"{text!r:30} → {len(pieces)} 個 token：{pieces}")

**觀察**

- 繁體的「天氣」和簡體的「天气」，切法一樣嗎？為什麼會這樣？
- 英文 token 前面的空格去哪了？
- 試試看你自己的句子，例如你的名字、一句台語、一段程式碼。

## 2. 看下一個 token 的機率

模型真正的輸出，是詞彙表裡**每一個** token 的機率，也就是一個機率分佈 probability distribution。

In [ ]:
ids = tok("今天天氣很", return_tensors="pt").input_ids

with torch.no_grad():                       # 只使用模型、不訓練，不用算梯度
    logits = model(ids).logits[0, -1]       # 第 0 句話、最後一個位置的分數，長度 = 詞彙表大小
probs = torch.softmax(logits, dim=-1)       # 分數 → 機率

top = torch.topk(probs, 10)
for p, i in zip(top.values, top.indices):
    print(f"{tok.decode([int(i)])!r:10} {p.item():.3f}")

print("前 10 名加總：", round(top.values.sum().item(), 3))
print("全部機率加總：", round(probs.sum().item(), 3))

**觀察**

- 前 10 名加起來大約多少？剩下的機率去哪了？
- 為什麼全部加起來一定是 1？

## 3. 自己寫生成迴圈

下面這幾行，就是聊天機器人產生回覆的核心迴圈：**每次只產生一個 token**，接回輸入，再整段重新送進模型。

In [ ]:
def generate(text, steps=20, temperature=1.0):
    ids = tok(text, return_tensors="pt").input_ids
    for _ in range(steps):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]
        if temperature == 0:
            next_id = torch.argmax(logits)                  # 貪婪解碼 greedy decoding：永遠選最高
        else:
            probs = torch.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, 1)[0]        # 依機率抽一個
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)   # 接回輸入尾端
        if next_id.item() == tok.eos_token_id:              # 產生「結束」token 就停
            break
    return tok.decode(ids[0])

In [ ]:
torch.manual_seed(0)  # 固定亂數種子，結果才能重現

for t in [0, 0.7, 1.5]:
    print(f"--- 溫度 T = {t}")
    for _ in range(3):
        print(generate("今天天氣很", steps=20, temperature=t))

**觀察**

- $T = 0$ 時，三次的結果一樣嗎？
- $T$ 越大，結果有什麼變化？

### 弄壞它：把溫度調到 5

先猜猜會發生什麼，再執行下一格。

In [ ]:
for _ in range(3):
    print(generate("今天天氣很", steps=20, temperature=5))

每個分數都先除以 $T$。$T$ 很大時，分數之間的差距被縮小成原來的 $1/T$，機率變得接近平均；十幾萬個 token 裡大多數都是不相干的字，所以幾乎每次都抽到不相干的字。

## 4. 偷看殘差流 residual stream

每個 Transformer 區塊都把結果「加回」殘差流，所以殘差流在每一層的形狀都一樣。我們可以把每一層之後的殘差流拿出來看。

In [ ]:
ids = tok("今天天氣很", return_tensors="pt").input_ids

with torch.no_grad():
    out = model(ids, output_hidden_states=True)
hs = out.hidden_states   # 嵌入的輸出＋每一層之後的殘差流

print("共", len(hs), "個快照")
for layer in [0, 1, len(hs) - 1]:
    print(f"第 {layer:2d} 個快照的形狀：{tuple(hs[layer].shape)}")

**觀察**：快照數量為什麼比層數多 1？形狀 `(1, 4, 896)` 的三個數字各代表什麼？

接著用 logit lens：把每一層的向量直接接到最後的「反嵌入 → Softmax」，看看模型「如果現在就要回答，最想接哪個字」。

In [ ]:
with torch.no_grad():
    for layer, h in enumerate(hs):
        x = h[0, -1]                         # 最後一個位置的向量
        if layer < len(hs) - 1:              # 最後一個快照已經做過最後的正規化，其他的要自己做
            x = model.model.norm(x)
        p = torch.softmax(model.lm_head(x), dim=-1)
        best = int(p.argmax())
        print(f"第 {layer:2d} 層之後，最想接：{tok.decode([best])!r:10} ({p[best].item():.2f})")

**觀察**：從第幾層開始，「最想接的字」變得合理？

> 中間層的猜測不一定準，logit lens 本來就是一種近似。讀不出答案，不等於那一層沒有資訊。

## 5.（選做）聊天模型真正看到的輸入

聊天模型會先把對話套上對話格式 chat template。看看它實際收到的是什麼：

In [ ]:
chat_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
msgs = [{"role": "user", "content": "今天天氣很"}]
text = chat_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
print(text)
print("token 數：", len(chat_tok(text).input_ids))

**觀察**：除了你輸入的「今天天氣很」，還多了哪些東西？token 數比 4 多了多少？

---

回到網站：[EP00 圖文教學](https://chainchen1103.github.io/llm-from-scratch/episodes/ep00/)